# GCP MLOps Complete Workflow

End-to-end guide for deploying the MLOps platform on Google Cloud Platform.

## Workflow

| Step | Folder | Description |
|------|--------|-------------|
| 1 | `01-gcs-storage-setup/` | Create GCS bucket, upload datasets |
| 2 | `02-model-training/` | Train TinyBERT + ViT models, push to GCS |
| 3 | `03-gce-compute-setup/` | Provision GCE instance |
| 4 | `04-local-app-development/` | FastAPI + Streamlit local dev |
| 5 | `05-deploy-streamlit-gce/` | Deploy Streamlit to GCE |
| 6 | `06-dockerize-app-and-fastapi/` | Containerize FastAPI + Nginx |
| 7 | `07-deploy-cloud-run/` | Push image to GAR, deploy Cloud Run |

**Python 3.11.8 throughout.**

In [ ]:
# Install all GCP SDK dependencies
!pip install \
    "google-cloud-storage>=2.14.0,<3.0.0" \
    "google-cloud-compute>=1.14.0,<2.0.0" \
    "google-cloud-artifact-registry>=1.10.0,<2.0.0" \
    "google-cloud-run>=0.10.0,<1.0.0" \
    "google-cloud-iam>=2.15.0,<3.0.0" \
    "google-api-python-client>=2.100.0,<3.0.0" \
    "python-dotenv>=1.0.0,<2.0.0" \
    "requests"

## Authentication

**Option A — Recommended for local development:**
```bash
gcloud auth application-default login
```
This stores credentials in `~/.config/gcloud/application_default_credentials.json`.
All Google Cloud client libraries pick this up automatically.

**Option B — Service Account Key:**
Download a JSON key from the GCP Console and set:
```bash
export GOOGLE_APPLICATION_CREDENTIALS=/path/to/key.json
```
Or add it to your `.env` file (copy from `.env.example`).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env in the current directory

PROJECT_ID = os.environ['GCP_PROJECT_ID']
REGION = os.environ.get('GCP_REGION', 'us-central1')
ZONE = os.environ.get('GCP_ZONE', 'us-central1-a')
GCS_BUCKET_NAME = os.environ['GCS_BUCKET_NAME']
GAR_REPO = os.environ.get('GAR_REPOSITORY_NAME', 'mlops-repo')
CLOUD_RUN_SERVICE = os.environ.get('CLOUD_RUN_SERVICE', 'mlops-api')
GCE_INSTANCE = os.environ.get('GCE_INSTANCE_NAME', 'mlops-prod')

SA_NAME = 'mlops-least-privilege'
SA_EMAIL = f'{SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com'

print(f'Project ID:  {PROJECT_ID}')
print(f'Region:      {REGION}')
print(f'GCS Bucket:  {GCS_BUCKET_NAME}')
print(f'SA Email:    {SA_EMAIL}')

## Step 1: Create Service Account (Least Privilege)

We create a dedicated service account for the MLOps application instead of using a personal account.
Roles are bound at the resource level (bucket, GAR repo) — not at project level — wherever possible.

| Role | Scope | Purpose |
|------|-------|---------|
| `roles/storage.objectViewer` | GCS bucket | Read model files |
| `roles/storage.objectCreator` | GCS bucket | Upload trained models |
| `roles/artifactregistry.writer` | GAR repository | Push Docker images |
| `roles/run.developer` | Project | Deploy Cloud Run services |

In [ ]:
from googleapiclient.discovery import build

iam_service = build('iam', 'v1')
sa_resource = f'projects/{PROJECT_ID}'

try:
    sa = iam_service.projects().serviceAccounts().get(
        name=f'projects/{PROJECT_ID}/serviceAccounts/{SA_EMAIL}'
    ).execute()
    print(f'Service account already exists: {sa["email"]}')
except Exception:
    sa = iam_service.projects().serviceAccounts().create(
        name=sa_resource,
        body={
            'accountId': SA_NAME,
            'serviceAccount': {
                'displayName': 'MLOps Least Privilege SA',
                'description': 'Service account for MLOps app — resource-scoped roles only',
            }
        }
    ).execute()
    print(f'Created service account: {sa["email"]}')

In [ ]:
# Bind roles to GCS bucket (resource-scoped — not project-wide)
from google.cloud import storage
from google.iam.v1 import policy_pb2

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET_NAME)

policy = bucket.get_iam_policy(requested_policy_version=3)

sa_member = f'serviceAccount:{SA_EMAIL}'
roles_to_add = ['roles/storage.objectViewer', 'roles/storage.objectCreator']

for role in roles_to_add:
    existing = next((b for b in policy.bindings if b['role'] == role), None)
    if existing:
        if sa_member not in existing['members']:
            existing['members'].add(sa_member)
            print(f'Added {SA_EMAIL} to existing binding for {role}')
        else:
            print(f'Already bound: {role}')
    else:
        policy.bindings.append({'role': role, 'members': {sa_member}})
        print(f'New binding created: {role}')

bucket.set_iam_policy(policy)
print(f'\nGCS bucket IAM policy updated for {GCS_BUCKET_NAME}')

In [ ]:
# Bind role to Artifact Registry repo (resource-scoped)
from google.cloud import artifactregistry_v1
from google.iam.v1 import iam_policy_pb2, policy_pb2

gar_client = artifactregistry_v1.ArtifactRegistryClient()
repo_path = f'projects/{PROJECT_ID}/locations/{REGION}/repositories/{GAR_REPO}'

get_req = iam_policy_pb2.GetIamPolicyRequest(resource=repo_path)
policy = gar_client.get_iam_policy(request=get_req)

gar_role = 'roles/artifactregistry.writer'
sa_member = f'serviceAccount:{SA_EMAIL}'

existing = next((b for b in policy.bindings if b.role == gar_role), None)
if existing:
    if sa_member not in existing.members:
        existing.members.append(sa_member)
else:
    policy.bindings.append(policy_pb2.Binding(role=gar_role, members=[sa_member]))

set_req = iam_policy_pb2.SetIamPolicyRequest(resource=repo_path, policy=policy)
gar_client.set_iam_policy(request=set_req)
print(f'GAR repo IAM updated: {gar_role} → {SA_EMAIL}')

In [ ]:
# Bind Cloud Run role at project level (no resource-level IAM for Cloud Run)
crm_service = build('cloudresourcemanager', 'v1')

policy = crm_service.projects().getIamPolicy(
    resource=PROJECT_ID, body={}
).execute()

run_role = 'roles/run.developer'
sa_member = f'serviceAccount:{SA_EMAIL}'

existing = next((b for b in policy.get('bindings', []) if b['role'] == run_role), None)
if existing:
    if sa_member not in existing['members']:
        existing['members'].append(sa_member)
        print(f'Added to existing binding: {run_role}')
    else:
        print(f'Already bound at project level: {run_role}')
else:
    policy.setdefault('bindings', []).append({'role': run_role, 'members': [sa_member]})
    print(f'Created project-level binding: {run_role}')

crm_service.projects().setIamPolicy(
    resource=PROJECT_ID,
    body={'policy': policy}
).execute()
print('Project IAM policy updated.')

## Generate Service Account Key

```bash
# Download a JSON key for the service account
gcloud iam service-accounts keys create ~/.gcloud/service-account-key.json \
    --iam-account mlops-least-privilege@YOUR_PROJECT_ID.iam.gserviceaccount.com

# Add to .gitignore!
echo '.gcloud/' >> .gitignore
```

> **Security note:** Never commit service account key files. For production, prefer
> attaching the SA directly to GCE instances or Cloud Run (no key file needed).

In [ ]:
# Step 1 (detailed): Create GCS bucket
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)

try:
    bucket = storage_client.get_bucket(GCS_BUCKET_NAME)
    print(f'Bucket already exists: gs://{GCS_BUCKET_NAME}')
except Exception:
    bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=REGION)
    print(f'Bucket created: gs://{GCS_BUCKET_NAME}')

print(f'Location: {bucket.location}')

In [ ]:
# GCS upload/download test
import io

test_blob = bucket.blob('_test/hello.txt')
test_blob.upload_from_string('Hello from MLOps!')
print('Upload OK')

content = test_blob.download_as_text()
print(f'Download OK: {content!r}')

test_blob.delete()
print('Test blob cleaned up')

In [ ]:
# Step 3: Create GCE instance with the MLOps service account attached
from google.cloud import compute_v1

instances_client = compute_v1.InstancesClient()

existing_instances = [i.name for i in instances_client.list(project=PROJECT_ID, zone=ZONE)]
if GCE_INSTANCE in existing_instances:
    print(f'Instance already exists: {GCE_INSTANCE}')
else:
    instance = compute_v1.Instance(
        name=GCE_INSTANCE,
        machine_type=f'zones/{ZONE}/machineTypes/n1-standard-2',
        disks=[
            compute_v1.AttachedDisk(
                boot=True,
                auto_delete=True,
                initialize_params=compute_v1.AttachedDiskInitializeParams(
                    source_image='projects/debian-cloud/global/images/family/debian-12',
                    disk_size_gb=50,
                ),
            )
        ],
        network_interfaces=[
            compute_v1.NetworkInterface(
                name='global/networks/default',
                access_configs=[compute_v1.AccessConfig(name='External NAT', type_='ONE_TO_ONE_NAT')],
            )
        ],
        # Attach the service account — no key file needed on the instance
        service_accounts=[
            compute_v1.ServiceAccount(
                email=SA_EMAIL,
                scopes=['https://www.googleapis.com/auth/cloud-platform'],
            )
        ],
    )
    op = instances_client.insert(project=PROJECT_ID, zone=ZONE, instance_resource=instance)
    op.result()
    print(f'Instance created: {GCE_INSTANCE}')

## Steps 4–5: Local Development & Streamlit

```bash
# Step 4 — FastAPI locally
cd infrastructure/gcp/04-local-app-development/fastapi
pip install -r requirements.txt
uvicorn main:app --reload --port 5000

# Step 4 — Streamlit locally (in a second terminal)
cd infrastructure/gcp/04-local-app-development/streamlit
streamlit run app.py

# Step 5 — Deploy Streamlit to GCE
# See 05-deploy-streamlit-gce/gce-streamlit-deploy.ipynb
```

In [ ]:
# Step 6: Create Artifact Registry repository (idempotent)
gar_client = artifactregistry_v1.ArtifactRegistryClient()
parent = f'projects/{PROJECT_ID}/locations/{REGION}'
repo_name = f'{parent}/repositories/{GAR_REPO}'

try:
    existing = gar_client.get_repository(name=repo_name)
    print(f'Repository already exists: {existing.name}')
except Exception:
    repo = artifactregistry_v1.Repository(
        format_=artifactregistry_v1.Repository.Format.DOCKER,
        description='MLOps Docker image repository',
    )
    op = gar_client.create_repository(parent=parent, repository_id=GAR_REPO, repository=repo)
    result = op.result()
    print(f'Repository created: {result.name}')

IMAGE_URI = f'{REGION}-docker.pkg.dev/{PROJECT_ID}/{GAR_REPO}/{CLOUD_RUN_SERVICE}:latest'
print(f'Image URI: {IMAGE_URI}')

In [ ]:
# Step 6: Docker build/push commands
print('Run in your terminal (Docker must be running):')
print()
print(f'gcloud auth configure-docker {REGION}-docker.pkg.dev')
print(f'docker build -t {CLOUD_RUN_SERVICE} infrastructure/gcp/06-dockerize-app-and-fastapi/')
print(f'docker tag {CLOUD_RUN_SERVICE} {IMAGE_URI}')
print(f'docker push {IMAGE_URI}')

In [ ]:
# Step 7: Deploy to Cloud Run
from google.cloud import run_v2

run_client = run_v2.ServicesClient()
parent_location = f'projects/{PROJECT_ID}/locations/{REGION}'
service_name_full = f'{parent_location}/services/{CLOUD_RUN_SERVICE}'

service = run_v2.Service(
    template=run_v2.RevisionTemplate(
        service_account=SA_EMAIL,  # SA has objectViewer on the bucket
        containers=[
            run_v2.Container(
                image=IMAGE_URI,
                ports=[run_v2.ContainerPort(container_port=5000)],
                env=[
                    run_v2.EnvVar(name='GCS_BUCKET_NAME', value=GCS_BUCKET_NAME),
                ],
                resources=run_v2.ResourceRequirements(
                    limits={'memory': '2Gi', 'cpu': '2'},
                ),
            )
        ],
    ),
    ingress=run_v2.IngressTraffic.INGRESS_TRAFFIC_ALL,
)

try:
    run_client.get_service(name=service_name_full)
    print('Service exists — updating...')
    service.name = service_name_full
    op = run_client.update_service(service=service)
except Exception:
    print(f'Creating Cloud Run service {CLOUD_RUN_SERVICE}...')
    op = run_client.create_service(
        parent=parent_location,
        service=service,
        service_id=CLOUD_RUN_SERVICE,
    )

result = op.result()
SERVICE_URL = result.uri
print(f'Service URL: {SERVICE_URL}')

In [ ]:
# Verify endpoints
import requests, time

time.sleep(5)

# Health check
resp = requests.get(SERVICE_URL, timeout=30)
print(f'Health: {resp.status_code} — {resp.json()}')

# Pose classifier test
resp = requests.post(
    f'{SERVICE_URL}/api/v1/pose_classifier',
    json={'url': ['https://images.pexels.com/photos/1755385/pexels-photo-1755385.jpeg']},
    timeout=60,
)
print(f'Pose classifier: {resp.status_code}')
print(resp.json())

In [ ]:
# Teardown
# Uncomment to clean up resources:

# Delete Cloud Run service
# run_client.delete_service(name=service_name_full).result()

# Delete GCE instance
# instances_client.delete(project=PROJECT_ID, zone=ZONE, instance=GCE_INSTANCE).result()

# WARNING: Deleting the GCS bucket will remove all trained models
# bucket.delete(force=True)

print('Teardown skipped. Uncomment the lines above to clean up resources.')

## Workflow Summary

| Step | Folder | What Was Done |
|------|--------|---------------|
| 1 | `01-gcs-storage-setup/` | Created GCS bucket |
| 2 | `02-model-training/` | Trained models, pushed to GCS |
| 3 | `03-gce-compute-setup/` | Created GCE instance with SA attached |
| 4 | `04-local-app-development/` | Developed FastAPI + Streamlit locally |
| 5 | `05-deploy-streamlit-gce/` | Deployed Streamlit to GCE |
| 6 | `06-dockerize-app-and-fastapi/` | Containerized app, pushed to GAR |
| 7 | `07-deploy-cloud-run/` | Deployed serverless Cloud Run API |

## IAM Comparison: GCP vs AWS

| Aspect | GCP | AWS |
|--------|-----|-----|
| Identity type | Service Account | IAM Role + Instance Profile |
| Storage scope | `set_iam_policy()` on GCS bucket | Inline policy with specific S3 bucket ARN |
| Registry scope | `set_iam_policy()` on GAR repo | ECR resource-based policy |
| Compute scope | SA attached to GCE instance | Instance Profile attached to EC2 |
| Serverless scope | SA set on Cloud Run service; project-level role for `run.developer` | Task Execution Role on ECS task definition |